# Notebook 07  Core ML Foundations

Building on Notebooks 01–06, this notebook bridges mathematical foundations to **classical machine-learning algorithms**: how models are trained, evaluated, and deployed without evaluation leakage.

Outline:
$$
\boxed{
\text{ML framing}
\rightarrow
\text{Statistical learning}
\rightarrow
\text{Data splitting and leakage}
\rightarrow
\text{Preprocessing}
\rightarrow
\text{Losses and metrics}
\rightarrow
\text{Generalization and selection}
\rightarrow
\text{Ready for classical algorithms}
}
$$

**Style note:** concepts stay short; experiments carry the weight. Likelihood, MLE, and information theory from Notebooks 04–06 are referenced rather than re-derived.


In [ ]:
import sys
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
from collections import Counter

from sklearn.datasets import make_classification, make_regression, load_breast_cancer
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GroupShuffleSplit, GroupKFold,
    cross_val_score, GridSearchCV, RandomizedSearchCV, KFold,
)
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, OrdinalEncoder, PolynomialFeatures,
    FunctionTransformer,
)
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    mean_squared_error, mean_absolute_error,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    classification_report, r2_score,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier, DummyRegressor

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.ml_foundations.assignments.loss_curves import interactive_huber_loss
from src.ml_foundations.assignments.cv_diagram import plot_kfold_diagram

np.set_printoptions(precision=4, suppress=True)
plt.rcParams['figure.dpi'] = 110
warnings.filterwarnings('ignore', category=FutureWarning)
RNG = np.random.default_rng(42)


## 1. What Machine Learning Is

Machine learning is the process of using **data** and an **objective** to find parameters that let a model make useful predictions on **unseen** data.

$$
X \xrightarrow{\; f_\theta \;} \hat y
\qquad\qquad
\theta^* = \arg\min_\theta \; \mathcal L(\theta; \text{data})
$$

| Term | Role |
|------|------|
| **Training** | Estimate parameters $\theta$ from labeled (or unlabeled) data |
| **Inference** | Apply the fitted model to new inputs |
| **Parameters** | Quantities learned from data (weights, splits, etc.) |
| **Objective / loss** | Scalar score minimized (or maximized) during training |
| **Prediction** | Model output on new $X$ |

Mitchell's definition: *"A computer program is said to learn from experience $E$ with respect to some class of tasks $T$ and performance measure $P$ if its performance at tasks in $T$, as measured by $P$, improves with experience $E$."*

This is useful because it forces to name the **task**, the **experience**, and the **metric**  three things practitioners often leave implicit.

Not every method is supervised prediction. **PCA** (Notebook 02) learns a low-dimensional representation by maximizing variance  no labeled target. It still fits the broader pattern: use data to learn parameters of a transformation, then apply that transformation to new observations.


In [ ]:
# Minimal supervised learning loop
X, y = make_regression(n_samples=200, n_features=3, noise=8, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)

model = LinearRegression()
model.fit(X_tr, y_tr)
pred = model.predict(X_te)
print(f"Learned coefficients: {model.coef_.round(3)}")
print(f"Test R²: {r2_score(y_te, pred):.3f}")
print(f"Test MSE: {mean_squared_error(y_te, pred):.2f}")


**Observation:** training found slope coefficients; inference applies them to held-out rows. The loss (MSE here) is what training minimized  not necessarily the quantity of interest in production.

For regression, **accuracy is not informative**: a continuous target has infinite precision, so the probability of predicting the exact value is essentially zero. **R²** measures improvement over predicting the mean: $R^2 = 1 - \frac{\sum_i (y_i - \hat y_i)^2}{\sum_i (y_i - \bar y)^2}$. An $R^2$ near 1 means the model explains most variance; near 0 means performance is close to the mean predictor. With synthetic Gaussian noise of standard deviation $\sigma$, test MSE is approximately $\sigma^2$ when the model fits well.


## 2. Learning Paradigms and Problem Types

| Paradigm | Signal | Example |
|----------|--------|---------|
| **Supervised** | Labeled $(X, y)$ pairs | Spam detection |
| **Unsupervised** | Structure in $X$ alone | Customer segmentation |
| **Semi-supervised** | Few labels + many unlabeled | 500 labeled images + 50k unlabeled |
| **Self-supervised** | Labels derived from data itself | Predict masked words |
| **Reinforcement** | Reward from environment interaction | Game-playing agent |

**Semi-supervised nuance:** "partially labeled" $\neq$ multiclass. The point is exploiting unlabeled data (consistency regularization, pseudo-labels)  not merely having multiple classes.

**Reinforcement learning:** state $\rightarrow$ action $\rightarrow$ environment $\rightarrow$ reward $\rightarrow$ update. The learning signal comes from interaction, not a static $(X,y)$ table. (Unlike LRTA\*, which is classical search with heuristic updates, RL learns a policy from accumulated reward.)

**Problem types** (one example each):

| Type | Example |
|------|---------|
| Regression | Predict house price |
| Binary classification | Fraud vs legitimate |
| Multiclass | Digit recognition (0–9) |
| Multilabel | Tag an article with multiple topics |
| Ranking | Order search results |
| Clustering | Group shoppers by behavior |
| Dimensionality reduction | Compress gene expression profiles |
| Density estimation | Model distribution of network traffic |
| Anomaly detection | Flag unusual sensor readings |


## 3. Statistical Learning Framework

Learning is framed as choosing a function $f$ from a hypothesis space $\mathcal F$ using a finite sample. The objects below connect theory to what code actually optimizes.

| Object | Meaning |
|--------|---------|
| **Population / DGP** | The true data-generating process; never fully observable |
| **Sample** | Finite observations $\{(x_i, y_i)\}_{i=1}^n$ drawn from the DGP |
| **Hypothesis space** $\mathcal F$ | Set of candidate models (e.g. all linear functions, all depth-5 trees) |
| **Loss** $\mathcal L(y, \hat y)$ | Per-example penalty for prediction error |
| **Population risk** | $R(f) = \mathbb E[\mathcal L(Y, f(X))]$  expected loss on new data |
| **Empirical risk** | $\hat R(f) = \frac1n \sum_i \mathcal L(y_i, f(x_i))$  average loss on the sample |

**Empirical risk minimization (ERM):**
$$\hat f = \arg\min_{f \in \mathcal F} \hat R(f)$$

Training minimizes $\hat R(f)$ because $R(f)$ cannot be computed directly: the expectation requires the unknown population distribution.

**Generalization gap:** for a fitted model $\hat f$,
$$\text{gap} =  R(\hat f) - \hat R(\hat f) $$
A small training loss with a large gap means the model **memorized** the sample but fails on new data. Capacity of $\mathcal F$ matters: a very flexible $\mathcal F$ can drive $\hat R(\hat f)$ to zero while $R(\hat f)$ stays high.

**What ERM does and does not promise**

- ERM finds the best fit **on the observed rows**.
- It does **not** guarantee best performance on unseen rows.
- Regularization, validation, and simpler model classes are ways to control the gap.

The experiment below uses a high-dimensional logistic model with weak regularization  a setting where ERM can fit the training set almost perfectly while test performance degrades.


In [ ]:
## High-dimensional classification: train great, test mediocre
X, y = make_classification(
    n_samples=300, n_features=200, n_informative=10,
    n_redundant=2, random_state=7,
)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.35, random_state=10, stratify=y)

model = LogisticRegression(max_iter=3000, C=1e6)  # c means inverse of regularization strength. large C -> weak regularization -> flexible fit
model.fit(X_tr, y_tr)

train_acc = accuracy_score(y_tr, model.predict(X_tr))
test_acc = accuracy_score(y_te, model.predict(X_te))
gap = train_acc - test_acc

fig, ax = plt.subplots(figsize=(5, 5))
bars = ax.bar(["Train", "Test"], [train_acc, test_acc], color=["#123456", "#abcdef"])
ax.set_ylim(0, 1.15)
ax.set_ylabel("Accuracy")
ax.set_title(f"ERM on flexible model\nGeneralization gap = {gap:.3f}")
for b, v in zip(bars, [train_acc, test_acc]):
    ax.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.3f}", ha="center")
plt.show()

**Interpretation:** near-perfect training accuracy with weaker test performance is the signature of **overfitting to the empirical risk**. The model exploited idiosyncrasies of the training sample.


## 4. IID Assumption and Distribution Shift

Most train/test theory assumes draws are **IID** — **independent** and **identically distributed** from a fixed joint distribution $P(X, Y)$.

- *Independent:* knowing one row does not change the distribution of another (time series, repeated patients, and duplicate rows violate this).
- *Identically distributed:* every row follows the same $P(X, Y)$ (deployment drift violates this).

Evaluation estimates population risk only when the test set reflects **future deployment conditions**. When that fails, metrics can be optimistic or pessimistic for the wrong reasons.

### Types of shift

| Shift | What is stable | What changes | Notation (informal) | Example |
|-------|----------------|--------------|---------------------|---------|
| **Covariate** | $P(Y|X)$ | $P(X)$ | train and test inputs differ | New hospital, same disease markers |
| **Label** | $P(X|Y)$ | $P(Y)$ | class prevalence differs | Rare disease becomes common |
| **Concept** | $P(X)$ may be stable | $P(Y|X)$ | decision boundary changes | Fraud patterns evolve |

**Covariate shift:**  the features (X) change, but the conditional probability of the label given those features (P(Y|X)) stays exactly the same. The underlying physics, biology, or logic connecting (X) to (Y) does not change; you are simply looking at a different population or subset of data.

- **Deep Dive Example (The Hospital):** Imagine you train an AI model to detect pneumonia using X-ray features (X) at a high-end private clinic.
    - In Training: The patients are generally younger, healthier, and have fewer severe lung artifacts.
    - In Testing: You deploy the model to a public hospital where patients are much older and have more advanced diseases. The distribution of features (X) has shifted massively.
    - Why it's Covariate Shift: If a 30-year-old at the private clinic and a 30-year-old at the public hospital have the exact same lung opacity feature, they have the exact same probability (P(Y|X)) of having pneumonia. The rule hasn't changed—just the demographic of people walking through the door.
- **The Danger:** The model might be forced to make predictions in "unfamiliar regions" of the feature space where it has low confidence or sparse training data, leading to unpredictable errors.


**Label shift:**  the overall prevalence of the classes (Y) changes, but the way those classes manifest physically (P(X|Y)) remains completely identical. It is essentially the reverse of covariate shift.
- **Deep Dive Example (The Epidemic):** Imagine training a model to diagnose Flu (\(Y\)) based on body temperature (\(X\)).
    - In Training: You collect data in August. The base rate of the flu is very low—maybe 1% of patients with a cough have it P(Y).
    - In Testing: Winter arrives, or a sudden pandemic hits. Now, 40% of patients coming in have the flu. The label distribution has skyrocketed.
    - Why it's Label Shift: If you take a patient who definitely has the flu Y=1, their body temperature X will look the same whether it's August or January. The physical manifestation of the disease P(X|Y) is stable; there are just statistically way more sick people around.
- **The Danger:** Most machine learning models implicitly learn the baseline prevalence of classes from the training set. If a model thinks the flu is rare, it will bias its predictions toward "No Flu." If the base rate surges, the model will suffer from massive under-prediction (false negatives) unless adjusted using techniques like prior probability adjustment or checking the confusion matrix metrics.

**Concept shift (concept drift):** is the most dangerous of the three. Here, the distribution of features P(X) might look completely normal and unchanged, but the actual meaning of those features has changed. The definition of what constitutes a label Y has evolved over time, shifting the decision boundary.

- **Deep Dive Example (The Fraudster):** Imagine a fraud detection model trained to flag suspicious credit card transactions.
    - In Training: Fraudulent behavior (Y=1) is tied to buying expensive electronics at 3 AM (X).
    - In Testing: Banks catch on, so scammers change their behavior. Now, they blend in by buying $15 groceries at 2 PM. Meanwhile, legitimate users start buying expensive electronics at 3 AM due to a late-night online shopping sale.
    - Why it's Concept Shift: The feature inputs—buying an expensive item at 3 AM—look exactly the same as before P(X) is stable. However, that feature no longer points to fraud. The real-world mapping (P(Y|X)) has fundamentally altered.
- **Other Examples:** Macroeconomic changes (e.g., inflation changing what a "high salary" means) or language evolution (e.g., the word "sick" shifting from meaning "unwell" to meaning "cool" in sentiment analysis).
- **The Danger:** Because P(X) looks stable, basic data pipeline monitoring tools might not flag any anomalies. The data looks clean, yet the model will be systematically wrong and confidently hallucinating outdated predictions.


In [ ]:
# Independence: IID noise vs autocorrelated series
n = 500
iid = RNG.normal(0, 1, n)
ar1 = np.zeros(n)
for t in range(1, n):
    ar1[t] = 0.9 * ar1[t - 1] + RNG.normal(0, 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))

axes[0].plot(iid[:100], marker="o", ms=3)
axes[0].set_title("IID noise")
axes[1].plot(ar1[:100], marker="o", ms=3, color="coral")
axes[1].set_title("Autocorrelated series\n(independence violated)")

plt.show()

## 5. Train / Validation / Test

| Split | Purpose |
|-------|---------|
| **Training** | Learn model parameters $\theta$ |
| **Validation** | Compare models, tune hyperparameters, decide early stopping |
| **Test** | Final unbiased estimate **after** all development choices are frozen |

Repeatedly peeking at the test set leaks information into model selection:  causing **indirect test-set overfitting**, also known as **human-in-the-loop leakage**.

To safely apply learning without leakage we should we should divide the dataset into a **development (training and validating) set** and **training set**, and we should **never** check the training set until we are **100% complete with our model**. When we are training we use validating set only to tweak the settings and parameters of the experiment and we will indirectly leak validating set to the model but that is okay because if the performance improved only because of that reason then the test set will expose the truth. If our dataset is small we can perform nested cross validation but we still need to keep the test set isolated. 

**What should we do if we run against the test set at the very end and it performs poorly?** If we use any insight we gain from this test set result to modify the model, then the test set will be polluted. One approach is to apply this finding and then collect new data that will serve as the fresh test set for the model. If we can't get access to new unseen data, then in one way or another, any tweaking or modification that we do after this will pollute the model with test set information, so we need to include that constraint in our final report if we decide to proceed with it. One way we can continue after this contamination is to run cross-validation on the whole dataset and report the average cross-validation score instead of a score against a standalone test set. 

There is **no universal 70/15/15 rule**. Split sizes depend on **dataset size, label noise, evaluation cost, and whether cross-validation** is used instead of a single validation fold. When we have a large amount of data, we might use more than 95–99% of the dataset for training, and we would still be left with large enough data volumes for testing and validating. When dealing with rare cases, we must ensure that our class proportions are identical across all splits, so we should apply stratified splitting instead of a naive random one.

## 6. Data Splitting Strategies


### 6.1 Random splitting

`random_state` fixes the pseudo-random permutation so splits are **reproducible**  same seed, same partition every run.


In [ ]:
X, y = make_classification(n_samples=1000, weights=[0.9, 0.1], random_state=1)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
print("Train positive rate:", y_tr.mean().round(3))
print("Test positive rate:", y_te.mean().round(3))


### 6.2 Stratified splitting

Preserves class proportions  useful for **imbalanced** classification, not mandatory for every task.


In [ ]:
# Compare random vs stratified on imbalanced data
results = []
for seed in range(30):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.15, random_state=seed)
    results.append(("random", seed, y_te.mean()))
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.15, random_state=seed, stratify=y)
    results.append(("stratified", seed, y_te.mean()))

df_splits = pd.DataFrame(results, columns=["method", "seed", "test_pos_rate"])
summary = df_splits.groupby("method")["test_pos_rate"].agg(["mean", "std"])
print(summary)
print(f"True prevalence: {y.mean():.3f}")


### 6.3 Time-based splitting

When rows have a natural time order (sales, sensor readings, logs), the deployment rule is: **train on the past, evaluate on the future**. Random row splits are invalid because they place future observations into training.

**Protocol:** sort by timestamp, use the earliest block for training and the latest block for validation/test. For repeated model development, use **rolling** or **expanding** windows rather than reshuffling history.

**Common mistakes:** computing rolling statistics on the full series before splitting; tuning hyperparameters on a shuffled split of time-ordered data; treating a higher random-split score as evidence that temporal splitting is unnecessary.

A dedicated temporal experiment is omitted here; the split strategy should match how the model will be used at deployment time.


### 6.4 Group-based splitting

When multiple rows belong to the same entity (patient, user, device), random row splits put the same entity in train and test.


In [ ]:
n_patients, visits_per = 40, 8
patient_ids = np.repeat(np.arange(n_patients), visits_per)
X_g = RNG.normal(size=(n_patients * visits_per, 5))
y_g = (patient_ids % 3 == 0).astype(int)

# Wrong: random row split
X_tr, X_te, y_tr, y_te, p_tr, p_te = train_test_split(
    X_g, y_g, patient_ids, test_size=0.25, random_state=0,
)
overlap = len(set(p_tr) & set(p_te))
print(f"Patients appearing in BOTH train and test (random split): {overlap}")

# Correct: group split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
tr_idx, te_idx = next(gss.split(X_g, y_g, groups=patient_ids))
print(f"Overlap with GroupShuffleSplit: {len(set(patient_ids[tr_idx]) & set(patient_ids[te_idx]))}")


### 6.5 Duplicate contamination

Near-duplicate or **exact duplicate** rows split across train and test make evaluation optimistic: the model may have already seen the same feature vector with the same label.

The experiment duplicates every row, then applies a random split. Some test rows have an identical copy in training. **1-nearest neighbor** is used here because it memorizes exact matches, making the contamination effect visible.

In [ ]:
X_unique, y_unique = make_classification(
    n_samples=120, n_features=10, n_informative=5, random_state=3,
)

# Contaminated: every row duplicated; random split can separate copies across train/test
X_dup = np.vstack([X_unique, X_unique])
y_dup = np.concatenate([y_unique, y_unique])
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(
    X_dup, y_dup, test_size=0.3, random_state=0, stratify=y_dup,
)

# Clean reference: unique rows only, same seed
X_tr, X_te, y_tr, y_te = train_test_split(
    X_unique, y_unique, test_size=0.3, random_state=0, stratify=y_unique,
)

from sklearn.neighbors import KNeighborsClassifier

knn_dup = KNeighborsClassifier(n_neighbors=1)
knn_dup.fit(X_tr_d, y_tr_d)
acc_knn_dup = accuracy_score(y_te_d, knn_dup.predict(X_te_d))

knn_clean = KNeighborsClassifier(n_neighbors=1)
knn_clean.fit(X_tr, y_tr)
acc_knn_clean = accuracy_score(y_te, knn_clean.predict(X_te))

contaminated_idx = [ i for i, row in enumerate(X_te_d) if any(np.allclose(row, tr_row) for tr_row in X_tr_d) ]
acc_memorized_rows = accuracy_score(y_te_d[contaminated_idx], pred_knn[contaminated_idx])

print(f"Test rows with a train duplicate: {len(contaminated_idx)} / {len(X_te_d)}")
print(f"1-NN on duplicate split:          {acc_knn_dup:.3f}")
print(f"  accuracy on memorized rows:     {acc_memorized_rows:.3f}")
print(f"1-NN on deduplicated split:       {acc_knn_clean:.3f}")


**Observation:** accuracy on **contaminated** test rows (those with a train duplicate) is typically much higher than on unique test rows in the same split. That gap reveals memorization, not generalization. Deduplication should happen **before** splitting when exact or near-exact repeats exist.

## 7. Data Leakage

**Leakage** = information that would not be available at prediction time enters training or evaluation, producing **optimistic** metrics.

Four common forms are demonstrated below, including feature-selection leakage demonstrations.


### 7.1 Target leakage

Including a post-diagnosis treatment column when predicting disease.


In [ ]:
# Synthetic patient data
n = 400
has_disease = RNG.integers(0, 2, n)
treatment = np.where(has_disease, 1, RNG.integers(0, 2, n))  # treatment AFTER diagnosis
age = RNG.normal(50, 10, n)

# WRONG feature set
X_wrong = np.column_stack([age, treatment])
# CORRECT feature set
X_correct = age.reshape(-1, 1)

def eval_split(X, y):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
    m = LogisticRegression(max_iter=1000)
    m.fit(X_tr, y_tr)
    return accuracy_score(y_te, m.predict(X_te))

print("With treatment (leaky):", eval_split(X_wrong, has_disease))
print("Without treatment:", eval_split(X_correct, has_disease))


### 7.2 Train-test contamination before splitting

Fitting preprocessing on the full dataset passes test statistics into training.


In [ ]:
# 1. Generate the standard dataset
X, y = make_classification(n_samples=600, n_features=5, random_state=22)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.15, random_state=22)

# 2. Inject extreme out-of-distribution values ONLY into the test set 
# This simulates a real-world scenario where the test/production data has unexpected noise.
X_te_corrupted = X_te.copy()
X_te_corrupted[:, 0] = X_te_corrupted[:, 0] * 50  # Amplify a feature in test data

# --- WRONG WAY (Data Leakage) ---
# The scaler sees the extreme test values and adjusts the global variance, 
# artificially shrinking the training features to accommodate them.
scaler_bad = StandardScaler().fit(np.vstack([X_tr, X_te_corrupted])) 
X_tr_bad = scaler_bad.transform(X_tr)
X_te_bad = scaler_bad.transform(X_te_corrupted)

# --- CORRECT WAY (No Leakage) ---
# The scaler only knows about training distributions.
scaler_good = StandardScaler().fit(X_tr)
X_tr_good = scaler_good.transform(X_tr)
X_te_good = scaler_good.transform(X_te_corrupted)

# Evaluate
m = LogisticRegression(max_iter=1000)

m.fit(X_tr_bad, y_tr)
acc_bad = accuracy_score(y_te, m.predict(X_te_bad))

m.fit(X_tr_good, y_tr)
acc_good = accuracy_score(y_te, m.predict(X_te_good))

print(f"Scale-on-all-data accuracy: {acc_bad:.3f}")
print(f"Scale-on-train-only accuracy: {acc_good:.3f}")

### 7.3 Preprocessing leakage (fit_transform before split)

The anti-pattern:
```python
X_scaled = scaler.fit_transform(X) # uses test statistics
X_train, X_test = train_test_split(X_scaled, ...)
```

The fix: **split first**, then `fit` on train, `transform` all splits.


### 7.4 Feature selection leakage

Selecting features on the full dataset lets test labels influence which features survive  even without an explicit `.fit()` on the test rows.


In [ ]:
# Feature-selection leakage demonstration
X, y = make_classification(
    n_samples=10000, n_features=500, n_informative=15,
    n_redundant=5, random_state=40,
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y,
)

# CORRECT: select on train only
selector = SelectKBest(score_func=f_classif, k=20)
X_train_sel = selector.fit_transform(X_train, y_train)
X_test_sel = selector.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_sel, y_train)
acc_correct = accuracy_score(y_test, model.predict(X_test_sel))

# WRONG: select using all data (including future test rows)
X_selected = SelectKBest(score_func=f_classif, k=20).fit_transform(X, y)
X_tr_bad, X_te_bad, y_tr_bad, y_te_bad = train_test_split(
    X_selected, y, test_size=0.3, random_state=40, stratify=y,
)
model_bad = LogisticRegression(max_iter=1000)
model_bad.fit(X_tr_bad, y_tr_bad)
acc_leaky = accuracy_score(y_te_bad, model_bad.predict(X_te_bad))

# Pipeline keeps selection inside train folds
pipe = Pipeline([
    ("selection", SelectKBest(score_func=f_classif, k=20)),
    ("model", LogisticRegression(max_iter=1000)),
])
pipe.fit(X_train, y_train)
acc_pipe = accuracy_score(y_test, pipe.predict(X_test))

print(f"Select on train only:  {acc_correct:.3f}")
print(f"Select on all data:    {acc_leaky:.3f}  (optimistic)")
print(f"Pipeline:              {acc_pipe:.3f}")


**Why it happens:** feature scoring uses $y$ for every row. Test labels participate in ranking features, so the model indirectly trains on test information.

### 7.5 Why leakage matters at deployment

Leakage $\rightarrow$ pretty validation numbers $\rightarrow$ confident deployment $\rightarrow$ real-world failure.


In [ ]:
# Target copied into features (extreme leakage)
X_clean = RNG.normal(size=(500, 5))
y_bin = RNG.integers(0, 2, 500)
X_leaky = np.column_stack([X_clean, y_bin])  # label smuggled into X

X_tr, X_te, y_tr, y_te = train_test_split(X_leaky, y_bin, test_size=0.3, random_state=42, stratify=y_bin)
m = LogisticRegression(max_iter=1000)
m.fit(X_tr, y_tr)
print(f"Leaked feature accuracy: {accuracy_score(y_te, m.predict(X_te)):.3f}")

X_tr, X_te, y_tr, y_te = train_test_split(X_clean, y_bin, test_size=0.3, random_state=42, stratify=y_bin)
m.fit(X_tr, y_tr)
print(f"Clean features accuracy: {accuracy_score(y_te, m.predict(X_te)):.3f}")


## 8. Missing Data

Missing Completely at Random, MCAR / Missing at Random, MAR /Missing not at random, MNAR describe the **missingness mechanism**, not the imputation method.

| Mechanism | Meaning |
|-----------|---------|
| **MCAR** | Missingness independent of observed and unobserved values |
| **MAR** | Missingness depends on observed variables only |
| **MNAR** | Missingness depends on the missing value itself or unobserved factors |

MNAR is hard: the missingness model generally cannot be learned from observed data alone.


1. MCAR: Missing Completely at Random
- What it means: The data is missing due to pure, random luck or an accident. The missingness has absolutely nothing to do with the data itself or any other variable.
- Example: A lab technician accidentally drops a test tube, or a sensor battery randomly dies for a few minutes.
- The Fix: This is the easiest to deal with. You can usually just safely drop these rows without biasing your model.

2. MAR: Missing at Random
- What it means: This name is a trap, it is not completely random. The data is missing, but the reason it's missing can be completely explained by another piece of information you already captured in your dataset.
- Example: You run a survey about mental health and income. Men are statistically less likely to answer questions about their income than women. The missing income data isn't random, but it is linked directly to the "Gender" column which you did collect.
- The Fix: Advanced imputation (filling in the blanks) based on the related columns.

3. MNAR: Missing Not at Random (The Dangerous One)
- What it means: The data is missing precisely because of the unobserved value itself. The system is actively hiding the truth from you.
- Example: You send a survey asking people, "Have you ever committed tax fraud?" The people who have committed fraud simply leave the question blank. The data is missing because the answer is "Yes."
- The Fix: Very difficult. Dropping rows or naive filling will completely destroy and bias your model.



In [ ]:
# MAR example: older patients skip income reporting more often
age = RNG.integers(20, 80, size=1000)
income = 30000 + 800 * age + RNG.normal(5000, 5000, 1000)
missing_prob = np.clip((age - 60) / 80, 0.05, 0.85)
missing = RNG.random(1000) < missing_prob

df_miss = pd.DataFrame({"age": age, "income": np.where(missing, np.nan, income)})
print(df_miss.groupby(pd.cut(df_miss["age"], bins=5))["income"].apply(lambda s: s.isna().mean()))


In [ ]:
# Compare simple imputation strategies
strategies = {
    "mean": SimpleImputer(strategy="mean"),
    "median": SimpleImputer(strategy="median"),
    "mode": SimpleImputer(strategy="most_frequent"),
}
fig, axes = plt.subplots(1, 4, figsize=(11, 3))
axes[0].hist(df_miss["income"].dropna(),bins=50,alpha=0.7)
axes[0].set_title("Initial")

for ax, (name, imp) in zip(axes[1:], strategies.items()):
    filled = imp.fit_transform(df_miss[["income"]]).ravel()
    ax.hist(filled, bins=50, alpha=0.7)
    ax.set_title(name)
plt.suptitle("Imputed income distributions (MAR data)")
plt.tight_layout()
plt.show()

# Missingness indicator can help models separate "missing" from "low value"
df_miss["income_missing"] = df_miss["income"].isna().astype(int)


## 9. Categorical Features

**Ordinal encoding** only when order is meaningful (`low < medium < high`). Arbitrary integers on nominal colors impose a false ordering.

**One-hot encoding** creates binary indicator columns per category.


In [ ]:
# One-hot encoding with unseen-category handling
names = [["Obssa"], ["Abiy"], ["Amin"], ["Sherif"], ["Wehib"], ["Barok"]]
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded = encoder.fit_transform(names)
for cat, row in zip(encoder.categories_[0], encoded):
    print(f"{cat:8s} -> {row.astype(int)}")


In [ ]:
# Unseen category at transform time
try:
    OrdinalEncoder().fit([["low"], ["medium"], ["high"]]).transform([["blue"]])
except ValueError as e:
    print("OrdinalEncoder rejects unseen labels:", e)

enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
enc.fit([["red"], ["blue"], ["green"]])
print("Unseen 'yellow':", enc.transform([["yellow"]]))


## 10. Feature Scaling

Feature A $\in [0,1]$, feature B $\in [0, 100000]$  distance-based methods (kNN, k-means) and gradient methods care about scale.

| Method | Formula | Notes |
|--------|---------|-------|
| Standardization | $(x-\mu)/\sigma$ | Zero mean, unit variance (if $\sigma>0$) |
| Min-max | $(x-x_{\min})/(x_{\max}-x_{\min})$ | Maps to $[0,1]$ on observed range |
| Robust | $(x-\text{median})/\text{IQR}$ | Outlier-resistant |

**Scale-sensitive:** kNN, SVM (typically), neural nets, k-means, regularized linear models with shared penalty.

**Less sensitive:** tree-based models (splits are rank-based).


In [ ]:
X = np.column_stack([
    RNG.uniform(0, 1, 200),
    RNG.uniform(0, 100000, 200),
])

X_outlier = X.copy()
X_outlier[0, 1] = 5e5

scalers = {
    "Standard": StandardScaler(),
    "MinMax": MinMaxScaler(),
    "Robust": RobustScaler(),
}

fig, axes = plt.subplots(1, 3, figsize=(12, 4))

for ax, (name, sc) in zip(axes, scalers.items()):
    Z = sc.fit_transform(X_outlier)
    
    # Passing the whole 2D matrix 'Z' plots both columns as distinct boxes
    ax.boxplot(Z, tick_labels=["Feature A", "Feature B"])
    ax.set_title(f"{name} Scaler")
    ax.grid(axis='y', linestyle='--', alpha=0.7)

plt.suptitle("Outlier Effect on Feature Distributions Across Scalers", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 11. Feature Transformations

Log / power transforms tame skew. Polynomial and interaction features expand what a **linear** model can represent:

$$x \mapsto [x, x^2, x^3], \qquad (x_1, x_2) \mapsto x_1 x_2$$

Feature engineering changes the representation  not the learning algorithm.


In [ ]:
# Linear model fails on quadratic data; polynomial features fix it
x = np.linspace(-3, 3, 200)
y = x**2 + RNG.normal(0, 0.5, 200)
X_lin = x.reshape(-1, 1)

lin = LinearRegression().fit(X_lin, y)
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X_lin)
quad = LinearRegression().fit(X_poly, y)

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(x, y, s=10, alpha=0.4, label="data")
ax.plot(x, lin.predict(X_lin), label="degree-1", lw=3)
ax.plot(x, quad.predict(X_poly), label="degree-2", lw=3)
ax.legend()
ax.set_title("Polynomial features make a linear model expressive")
plt.show()


## 12. Fit vs Transform vs Predict

| Method | Learns from data? | Output |
|--------|-------------------|--------|
| `fit` | Yes  stores parameters | self |
| `transform` | No  applies learned map | transformed $X$ |
| `predict` | No (after fit) | model outputs |

Test rows must be transformed with **training** statistics, not their own.


In [ ]:
X_demo = np.concatenate([
    RNG.normal(0, 1, 200),
    RNG.normal(10, 1, 200),
]).reshape(-1, 1)
X_train_demo, _ = train_test_split(X_demo, test_size=0.5, random_state=0)

scaler = StandardScaler()
scaler.fit(X_train_demo)
print("Train mean used for scaling:", scaler.mean_[0].round(3))
print("Transform x=5 using train stats:", scaler.transform([[5.0]])[0, 0].round(3))


## 13. Pipelines

After leakage has been demonstrated manually, pipelines bundle preprocessing + model so each CV fold refits preprocessing on training folds only.


In [ ]:
# Mixed-type synthetic dataset
df_pipe = pd.DataFrame({
    "age": RNG.integers(18, 80, 400).astype(float),
    "income": RNG.normal(50000, 15000, 400),
    "city": RNG.choice(["Addis", "Nairobi", "Cairo", "Lagos"], 400),
    "churn": RNG.integers(0, 2, 400),
})
df_pipe.loc[RNG.choice(400, 40), "income"] = np.nan

num_cols = ["age", "income"]
cat_cols = ["city"]
preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

full_pipe = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=2000)),
])

X = df_pipe.drop(columns="churn")
y = df_pipe["churn"]
scores = cross_val_score(full_pipe, X, y, cv=5, scoring="accuracy")
print(f"5-fold CV accuracy: {scores.mean():.3f} ± {scores.std():.3f}")


## 14. Loss Functions

A **loss function** $\mathcal L(y, \hat y)$ scores how far a prediction $\hat y$ is from the target $y$. Training minimizes the **empirical average loss** (Section 3). The loss is what optimization descends; it is not always the same as the **evaluation metric** (Section 16).

$$\theta^* = \arg\min_\theta \frac{1}{n}\sum_i \mathcal L(y_i, f_\theta(x_i))$$

### Regression losses

| Loss | Formula | Advantages | Disadvantages | Typical use |
|------|---------|------------|---------------|-------------|
| **MSE** (mean squared error) | $\frac1n\sum (y_i-\hat y_i)^2$ | Smooth, differentiable; linked to Gaussian MLE (Notebook 04) | Very sensitive to outliers | Default regression with roughly Gaussian noise |
| **RMSE** (root MSE) | $\sqrt{\text{MSE}}$ | Same units as $y$ | Same outlier sensitivity as MSE | Reporting; same model ranking as MSE on a fixed set |
| **MAE** (mean absolute error) | $\frac1n\sum |y_i-\hat y_i|$ | Robust to outliers | Non-smooth at $e=0$ | Heavy-tailed errors, outliers |
| **Huber** | Quadratic for small $|e|$, linear beyond $\delta$ | Robust + smooth near zero | Extra hyperparameter $\delta$ | Compromise between MSE and MAE |

Huber loss for residual $e = y - \hat y$:
$$
\mathcal L_\delta(e) = \begin{cases}
\frac12 e^2 & |e| \le \delta \\
\delta(|e| - \frac12\delta) & |e| > \delta
\end{cases}
$$


In [ ]:
p = np.linspace(0.01, 0.99, 200)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for y_true, label in [(1, "y=1"), (0, "y=0")]:
    loss = -(y_true * np.log(p) + (1 - y_true) * np.log(1 - p))
    ax[0].plot(p, loss, label=label, lw=2)
ax[0].set_xlabel("predicted probability ŷ")
ax[0].set_ylabel("BCE loss")
ax[0].legend()
ax[0].set_title("BCE vs confidence")

# Multiclass is a short mention: softmax + cross-entropy over K classes
ax[1].axis("off")
ax[1].text(0, 0.5, "Multiclass: one-hot y + softmax →\n−Σ_k y_k log ŷ_k", fontsize=12)
plt.tight_layout()
plt.show()


## 15. Metrics vs Loss

The **training loss** is what optimization minimizes. The **evaluation metric** judges usefulness. They can disagree.

### Confusion matrix (binary classification)

|  | Predicted negative | Predicted positive |
|--|------------------|--------------------|
| **Actual negative** | TN (true negative) | FP (false positive) |
| **Actual positive** | FN (false negative) | TP (true positive) |

### Core metrics

| Metric | Formula | Interpretation | When it matters |
|--------|---------|----------------|-----------------|
| **Accuracy** | $(TP+TN)/N$ | Fraction correct overall | Balanced classes |
| **Precision** | $TP/(TP+FP)$ | Of predicted positives, fraction correct | Cost of false alarms |
| **Recall** (sensitivity, TPR) | $TP/(TP+FN)$ | Of actual positives, fraction found | Cost of missed positives |
| **Specificity** | $TN/(TN+FP)$ | Of actual negatives, fraction found | Cost of missed negatives |
| **F1** | $2PR/(P+R)$ | Balance of precision and recall | Imbalanced data |
| **ROC-AUC** | Area under TPR vs FPR curve | Ranking quality across thresholds | Overall separability |
| **PR-AUC** (avg precision) | Area under precision-recall curve | Focus on positive class | Rare positives; more informative than ROC-AUC then |

**False Positive Rate (FPR)**: $ 1 - Specificity$ = $ FP/(TN+FP) $

**How to interpret:**

- High precision, low recall: few false alarms, many missed positives.
- Low precision, high recall: many false alarms, few missed positives.
- High accuracy + low F1: likely majority-class predictor on imbalanced data.
- ROC-AUC $\approx 0.5$: random ranking; $\approx 1$: strong ranking.
- Compare PR-AUC to **class prevalence** (always-negative baseline).


In [ ]:
X, y = make_classification(n_samples=2000, weights=[0.97, 0.03], random_state=11)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=11, stratify=y)

maj = DummyClassifier(strategy="most_frequent")
maj.fit(X_tr, y_tr)
print("Majority baseline accuracy:", accuracy_score(y_te, maj.predict(X_te)))

clf = LogisticRegression(max_iter=2000, class_weight="balanced")
clf.fit(X_tr, y_tr)
pred = clf.predict(X_te)
print(classification_report(y_te, pred, digits=3))


In [ ]:
# 1. Generate synthetic binary classification data
X, y = make_classification(n_samples=1000, n_features=10, random_state=42)

# 2. Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 3. Train Logistic Regression
log_model = LogisticRegression()
log_model.fit(X_train, y_train)
y_probs_log = log_model.predict_proba(X_test)[:, 1] # Probability of positive class

# 4. Train Linear Regression 
lin_model = LinearRegression()
lin_model.fit(X_train, y_train)
y_probs_lin = lin_model.predict(X_test) # Continuous raw predictions

# 5. Compute ROC metrics and AUC scores
fpr_log, tpr_log, _ = roc_curve(y_test, y_probs_log)
auc_log = roc_auc_score(y_test, y_probs_log)

fpr_lin, tpr_lin, _ = roc_curve(y_test, y_probs_lin)
auc_lin = roc_auc_score(y_test, y_probs_lin)

# 6. Plot the comparison graph
plt.figure(figsize=(8, 6))

# Plot Logistic Regression curve
plt.plot(fpr_log, tpr_log, color='darkorange', lw=2, 
         label=f'Logistic Regression (AUC = {auc_log:.3f})')

# Plot Linear Regression curve
plt.plot(fpr_lin, tpr_lin, color='seagreen', lw=2, 
         label=f'Linear Regression (AUC = {auc_lin:.3f})')

# Plot baseline random guess
plt.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Random Guess (AUC = 0.500)')

# Graph adjustments
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity / Recall)')
plt.title('ROC Curve Comparison: Logistic vs. Linear Regression')
plt.legend(loc="lower right")
plt.grid(True, linestyle=':', alpha=0.6)

# Display graph
plt.show()

## 16. Generalization, Overfitting, Underfitting

Vary model complexity (polynomial degree) and track train vs validation error.


In [ ]:
x = np.linspace(0, 1, 40)
y = np.sin(2 * np.pi * x) + RNG.normal(0, 0.3, 40) + np.cos(x+np.pi)
X = x.reshape(-1, 1)
degrees = range(1, 30)
train_err, val_err = [], []

for d in degrees:
    X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.3, random_state=0)
    pf = PolynomialFeatures(d, include_bias=False)
    Xtr_p = pf.fit_transform(X_tr)
    Xva_p = pf.transform(X_va)
    reg = LinearRegression().fit(Xtr_p, y_tr)
    train_err.append(mean_squared_error(y_tr, reg.predict(Xtr_p)))
    val_err.append(mean_squared_error(y_va, reg.predict(Xva_p)))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(degrees), train_err, marker="o", label="Train MSE")
ax.plot(list(degrees), val_err, marker="o", label="Val MSE")
ax.set_xlabel("Polynomial degree (complexity)")
ax.set_ylabel("MSE")
ax.legend()
ax.set_title("Underfitting → sweet spot → overfitting")
plt.show()


## 17. Regularization

### The problem

When the feature count is large or features are correlated, **ordinary least squares (OLS)** can overfit: coefficients grow large and unstable, training error falls, but test error rises (Section 16). **Regularization** adds a penalty on coefficient magnitude so the fit stays useful on unseen data.

### Penalized objective

Ridge and Lasso replace unconstrained least squares with:

$$\hat\theta = \arg\min_\theta \underbrace{\frac{1}{n}\sum_i (y_i - x_i^\top\theta)^2}_{\text{data fit (MSE)}} + \underbrace{\lambda \cdot \Omega(\theta)}_{\text{complexity penalty}}$$

$\lambda \geq 0$ is the **regularization strength** (called `alpha` in scikit-learn). **Larger $\lambda$** means a stronger penalty, smaller coefficients, a simpler model, and the usual bias–variance trade-off: more bias, less variance. If $\lambda$ is too large, the model underfits (all coefficients near zero). If $\lambda$ is too small, behavior is close to unregularized OLS.

### Connection to MAP (maximum a posteriori)

Notebook 04 derived **MLE** by maximizing $P(\text{data} \mid \theta)$, which is equivalent to minimizing negative log-likelihood (for example MSE under Gaussian noise).

**MAP** estimation adds a **prior** $P(\theta)$ on coefficients before seeing the data:

$$\hat\theta_{\text{MAP}} = \arg\max_\theta \, P(\theta \mid D) = \arg\max_\theta \, P(D \mid \theta)\, P(\theta)$$

Taking $-\log$ turns the prior into an additive penalty on top of the data term:

$$\hat\theta_{\text{MAP}} = \arg\min_\theta \left[ -\log P(D \mid \theta) - \log P(\theta) \right]$$

| Method | Prior on each $\theta_j$ (i.i.d.) | Penalty $\Omega(\theta)$ | MAP reading |
|--------|-----------------------------------|---------------------------|-------------|
| **OLS** | Uniform (no preference) | none | Pure MLE |
| **Ridge** (L2) | $\theta_j \sim \mathcal{N}(0, \tau^2)$ Gaussian | $\|\theta\|_2^2 = \sum_j \theta_j^2$ | Large coefficients are unlikely a priori |
| **Lasso** (L1) | $\theta_j \sim \mathrm{Laplace}(0, b)$ | $\|\theta\|_1 = \sum_j \lvert\theta_j\rvert$ | Coefficients are expected near zero; exact zero is most probable |
| **Elastic Net** | Mixture of Gaussian + Laplace | $\alpha\|\theta\|_1 + (1-\alpha)\|\theta\|_2^2$ | Sparsity plus stability when features correlate |

$\lambda$ controls how tight the prior is: **larger $\lambda$** corresponds to a **narrower** prior and stronger shrinkage toward zero. Ridge and Lasso are not different algorithms in kind; they are the same MAP recipe with **different priors**.

### Why L1 produces exact zeros but L2 does not (geometry)

An equivalent picture: minimize MSE while keeping coefficients inside a small region.

- **Ridge (L2 budget):** the feasible region is a **disk/sphere**. Contours of MSE are ellipses; the optimum usually meets the boundary at a point where **no coordinate is exactly zero**.
- **Lasso (L1 budget):** the feasible region is a **diamond** with **sharp corners on the axes**. The ellipse often touches a **corner**, forcing some $\theta_j = 0$ exactly.

So Lasso performs **continuous feature selection** as $\lambda$ grows: coefficients are deleted one by one. Ridge only **shrinks** magnitudes; every feature stays in the model with a reduced weight.

## 18. Cross-Validation

K-fold, stratified K-fold, group K-fold, and time-aware validation each respect different data structure. Random K-fold is wrong for grouped or temporal data.

CV must wrap the **entire pipeline** when preprocessing learns from data.


In [ ]:
plot_kfold_diagram(n_splits=5)


## 19. Hyperparameters and Model Selection

### Parameters vs hyperparameters

A model has two kinds of quantities:

| Kind | Symbol | When set | Examples |
|------|--------|----------|----------|
| **Parameters** | $\theta$ | During `fit`, by minimizing training loss | Regression weights, tree split rules, neural net weights |
| **Hyperparameters** | $h$ | **Before** training, by the practitioner or search procedure | $\lambda$, `C`, tree depth, $k$ in kNN, learning rate |

Training solves:

$$\hat\theta(h) = \arg\min_\theta \frac{1}{n_{\text{train}}} \sum_{i \in \text{train}} \mathcal L\bigl(y_i, f_\theta(x_i)\bigr)$$

The deployed model is $f_{\hat\theta(h)}$: changing $h$ changes the fitted parameters. Hyperparameters control **capacity**, **regularization**, and **optimization** — they are not updated by gradient descent on the training loss.

### What model selection optimizes

Model selection picks hyperparameters that minimize **validation error**, not training error:

$$h^* = \arg\min_{h \in \mathcal H} \; \widehat{\text{Err}}_{\text{val}}\bigl(f_{\hat\theta(h)}\bigr)$$

$\mathcal H$ is the search space (grid of candidate values). $\widehat{\text{Err}}_{\text{val}}$ is estimated on a **held-out validation set** or by **cross-validation** (Section 19). The **test set must never** guide the choice of $h$; it is reserved for a single final evaluation after $h^*$ is fixed.

### Train / validation / test roles

| Split | Used for | Must not be used for |
|-------|----------|----------------------|
| **Train** | Fit $\hat\theta(h)$ for each candidate $h$ | Choosing $h$ or reporting final performance |
| **Validation / CV** | Compare candidates and pick $h^*$ | Repeated peeking without a fresh holdout |
| **Test** | One unbiased estimate of performance at $h^*$ | Tuning $h$, feature engineering, or any model decision |

If the test set influences any modeling choice, reported test scores are **optimistically biased** (validation overfitting).

### Cross-validation score (review)

For $K$-fold CV with folds $\mathcal F_1, \ldots, \mathcal F_K$:

$$\widehat{\text{Err}}_{\text{CV}}(h) = \frac{1}{K} \sum_{k=1}^{K} \text{metric}\bigl(f_{\hat\theta_k(h)}, \mathcal F_k\bigr)$$

where $\hat\theta_k(h)$ is fit on all folds except $\mathcal F_k$. `GridSearchCV` and `RidgeCV` automate this loop.

### Search strategies

| Method | Procedure | Cost | When it helps |
|--------|-----------|------|---------------|
| **Manual** | Expert picks $h$ from domain knowledge | Low | Strong priors, prototyping |
| **Grid search** | Evaluate every point in a finite grid $\mathcal H = \{h_1, \ldots, h_M\}$ | $O(M \times K)$ CV fits | Few hyperparameters (1–3) |
| **Random search** | Sample $h \sim q(h)$ for a fixed budget $B$ | $O(B \times K)$ | Many dimensions; often finds good $h$ in fewer tries than a sparse grid |
| **Bayesian optimization** | Model $\widehat{\text{Err}}(h)$ with a surrogate, sample where improvement is likely | Adaptive | Expensive models (large neural nets, long simulations) |

**Grid search** is exhaustive on the grid but **not** on the true optimum unless the grid is fine enough. **Random search** explores more unique values per dimension when only a few hyperparameters matter (Bergstra & Bengio, 2012).

The first 3 methods are uninformed or memoryless baselines, while Bayesian is the informed strategy and best when dealing with expensive models. 

#### Bayesian Optimization 
1. Start with small number of sample points randomly chosen
2. Use these points to generate the **Surrogate Function**
3. Iterate in the loop
    - Add additional points to the sample points using **Acquisition Function**
    - Re-evaluate the surrogate function
    - Continue the loop until the surrogate max doesn't change, variance is below threshold or f gets exhausted

We pass the surrogate into a cheap mathematical formula called the Acquisition Function, which scores every candidate point by balancing two competing goals:
1. Exploitation: Sampling where the surrogate predicts the best performance (lowest mean error $\mu(h)$). zooming in on a promising valley we already found.
2. Exploration: Sampling where uncertainty $\sigma(h)$ is huge-checking an uncharted region just in case a much deeper valley is hiding there.

The most popular acquisition function is **Expected Improvement (EI)**, which calculates how much better a candidate point is expected to be compared to the best result seen so far ($h^+$). Another common choice is Lower Confidence Bound (LCB) (or Upper Confidence Bound when maximizing accuracy):$$\text{LCB}(h) = \mu(h) - \kappa \cdot \sigma(h)$$Here, $\kappa$ is a tunable weight: a small $\kappa$ forces the search to hug known good areas (exploitation), while a large $\kappa$ forces it to investigate high-uncertainty pockets (exploration).

### Nested cross-validation

When the goal is an **unbiased performance estimate** while also tuning $h$, use **nested CV**:

- **Outer loop:** estimate generalization (each fold is a pseudo-test set).
- **Inner loop:** run grid/random search on the outer training folds to pick $h^*$.

Tuning on the same validation fold repeatedly and then reporting that fold's score **inflates** the metric. Nested CV or a strictly held-out test set avoids this.

### Link to regularization: `C` vs $\lambda$

For `LogisticRegression` with L2 penalty (Section 18):

$$\min_\theta \; -\log P(D \mid \theta) + \frac{1}{C}\|\theta\|_2^2$$

**$C$ is the inverse regularization strength:** large $C$ $\rightarrow$ weak penalty $\rightarrow$ complex model; small $C$ $\rightarrow$ strong penalty $\rightarrow$ simple model. If the theoretical penalty is $\lambda \|\theta\|_2^2$, then $C = 1/\lambda$ (up to scaling conventions in the library).

### How to interpret grid-search output

- **`best_params_`:** $h^*$ with lowest mean CV score on training data.
- **`best_score_`:** mean CV score at $h^*$ — still an estimate on training folds, not the final test score.
- **`cv_results_`:** full table of scores per grid point; useful for seeing whether performance is flat (robust choice) or sharply peaked (fragile).

After $h^*$ is chosen, the model is refit on all training data and evaluated **once** on the test set.

Below: `GridSearchCV` over `C` in a logistic regression pipeline (scaling inside the pipeline).


In [ ]:
X_hp, y_hp = make_classification(n_samples=500, n_features=20, random_state=5)
X_hp_tr, X_hp_te, y_hp_tr, y_hp_te = train_test_split(
    X_hp, y_hp, test_size=0.3, random_state=5, stratify=y_hp,
)
param_grid = {"model__C": [0.01, 0.1, 1, 10]}
search = GridSearchCV(
    Pipeline([("scale", StandardScaler()), ("model", LogisticRegression(max_iter=2000))]),
    param_grid, cv=5, scoring="accuracy",
)
search.fit(X_hp_tr, y_hp_tr)

print("Best C:", search.best_params_)
print("Best CV accuracy:", round(search.best_score_, 3))
print("Test accuracy (single evaluation at h*):", round(search.score(X_hp_te, y_hp_te), 3))

# CV scores across the grid
for c, mean_score in zip(
    param_grid["model__C"],
    search.cv_results_["mean_test_score"],
):
    print(f"  C={c:5.2f}  CV accuracy={mean_score:.3f}")


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import KFold, GridSearchCV, cross_val_score
from sklearn.svm import SVC
import numpy as np

# 1. Load a sample dataset
X, y = load_breast_cancer(return_X_y=True)

# 2. Define the Inner and Outer CV splitters
inner_cv = KFold(n_splits=3, shuffle=True, random_state=42)
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)

# 3. Set up the Inner Loop (GridSearchCV tunes C and gamma on inner folds)
param_grid = {'C': [0.1, 1, 10, 100], 'gamma': [0.001, 0.01, 0.1, 1]}
inner_search = GridSearchCV(
    estimator=SVC(),
    param_grid=param_grid,
    cv=inner_cv,
    scoring='accuracy'
)

# 4. Run the Outer Loop (evaluates the tuned inner_search on each outer test fold)
nested_scores = cross_val_score(
    estimator=inner_search,
    X=X,
    y=y,
    cv=outer_cv,
    scoring='accuracy'
)

print(f"Unbiased Nested CV Accuracy: {np.mean(nested_scores):.4f} (+/- {np.std(nested_scores):.4f})")

## 20. Class Imbalance

### Definition

Let $\pi = P(y=1)$ be the **positive class prevalence**. Data is **imbalanced** when $\pi$ is far from $0.5$ (e.g. fraud detection with $\pi \approx 0.001$). The **imbalance ratio** is often reported as $(1-\pi)/\pi$ (negatives per positive).

### Why accuracy fails

A **majority-class** classifier that always predicts the common class achieves:

$$\text{Accuracy} = \max(\pi, 1-\pi)$$

With $\pi = 0.05$, always predicting negative gives **95% accuracy** while finding **zero** positives ($\text{Recall} = 0$). Accuracy treats all errors equally; when the minority class is the target, missed positives (FN) and false alarms (FP) have asymmetric cost (Section 16).

### Metrics to prefer (Section 16)

| Metric | Formula | What to look for under imbalance |
|--------|---------|----------------------------------|
| **Precision** | $TP/(TP+FP)$ | High when false alarms are costly |
| **Recall** | $TP/(TP+FN)$ | High when missed positives are costly |
| **F1** | $2PR/(P+R)$ | Single balance of P and R |
| **PR-AUC** | $\int \text{Precision}(\tau)\, d\,\text{Recall}(\tau)$ | Ranking quality for the positive class; compare to baseline $\pi$ |

**ROC-AUC** can stay high when negatives dominate even if precision on positives is poor; **PR-AUC** is usually more informative when $\pi$ is small.

### Decision threshold is a hyperparameter

Most classifiers (like Logistic Regression, Random Forests, or Neural Networks) do not naturally output a hard class label ($0$ or $1$). Instead, predict_proba outputs a continuous probability $\hat{p} = P(y=1 \mid x)$ between $0.0$ and $1.0$.When you call .predict(X) in scikit-learn, it silently applies a default cutoff rule:If $\hat{p} > 0.5 \rightarrow$ predict $1$ (Positive) If $\hat{p} \le 0.5 \rightarrow$ predict $0$ (Negative)

`predict_proba` returns $\hat p = P(y=1 \mid x)$. The default rule $\hat y = 1$ if $\hat p > 0.5$ is optimal only when:

- classes are balanced, and
- false positives and false negatives have **equal cost**.

For general costs $c_{FP}$ and $c_{FN}$, the cost-sensitive threshold is:

$$\tau^* = \frac{c_{FP}}{c_{FP} + c_{FN}} \cdot \frac{1-\pi}{\pi}$$

(derivation assumes calibrated probabilities and fixed costs). Lowering $\tau$ **increases recall** and usually **decreases precision**; raising $\tau$ does the opposite. Threshold tuning is done on **validation** data, not the test set.

### Class-weighted loss

`class_weight="balanced"` reweights the training loss so minority errors count more. For binary logistic regression, each sample is weighted by:

$$w_i = \frac{n}{2 \, n_{c_i}}$$

where $n_{c_i}$ is the count of class $c_i$. Equivalently, the weighted BCE is:

$$\mathcal L_w = -\sum_i w_i \bigl[ y_i \log \hat p_i + (1-y_i)\log(1-\hat p_i) \bigr]$$

This does **not** change the test distribution; it changes what the model is penalized for during training.

### Other strategies

| Strategy | Mechanism | Advantage | Risk |
|----------|-----------|-----------|------|
| **Stratified splits** | Train/val/test keep $\pi$ | Stable metric estimates | Does not create new minority examples |
| **Undersampling majority** | Drop majority rows | Faster training | Loses majority information |
| **Oversampling minority** | Duplicate or synthesize minority rows | More minority signal | Overfitting to duplicates (SMOTE deferred) |
| **Class weights** | Reweight loss | Simple, no resampling | Requires cost calibration at threshold |

Below: imbalanced data ($\pi \approx 0.05$), balanced class weights, default vs tuned threshold, and the precision–recall curve.


In [ ]:
X, y = make_classification(n_samples=3000, weights=[0.95, 0.05], random_state=9)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=9, stratify=y)
pi = y_te.mean()

clf = LogisticRegression(max_iter=3000, class_weight="balanced")
clf.fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]

# Majority baseline
maj_acc = max(pi, 1 - pi)
print(f"Class prevalence pi = {pi:.3f}  |  majority-class accuracy = {maj_acc:.3f}")

for tau in [0.2, 0.5, 0.8]:
    pred = (proba >= tau).astype(int)
    print(
        f"threshold={tau:.1f}  "
        f"precision={precision_score(y_te, pred, zero_division=0):.3f}  "
        f"recall={recall_score(y_te, pred, zero_division=0):.3f}  "
        f"F1={f1_score(y_te, pred, zero_division=0):.3f}"
    )

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_te, (proba > 0.5).astype(int), ax=axes[0])
axes[0].set_title("Default threshold 0.5")

prec, rec, thr = precision_recall_curve(y_te, proba)
axes[1].plot(rec, prec)
axes[1].axhline(pi, color="gray", ls="--", lw=1, label=f"baseline prevalence ({pi:.2f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title(f"PR curve (AP={average_precision_score(y_te, proba):.3f})")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()


## 21. Baselines and Experimental Methodology

### Why baselines are required

A **baseline** is the simplest reasonable predictor for the task. A learned model must **beat the baseline** on held-out data to justify added complexity, training cost, and maintenance. Without a baseline, it is unclear whether features or model structure carry any signal.

### Formal baselines

**Regression.** The **mean predictor** $\hat y = \bar y_{\text{train}}$ has test MSE:

$$\text{MSE}_{\text{baseline}} = \operatorname{Var}(y_{\text{test}}) + (\bar y_{\text{train}} - \bar y_{\text{test}})^2$$

When train and test come from the same distribution, the mean baseline MSE is approximately $\operatorname{Var}(y)$. **$R^2$** (Section 1) measures improvement over this:

$$R^2 = 1 - \frac{\text{MSE}_{\text{model}}}{\text{MSE}_{\text{baseline}}}$$

$R^2 \leq 0$ means the model is no better than predicting the mean.

**Classification.** The **majority-class** predictor always outputs the most frequent training class. Its accuracy is:

$$\text{Acc}_{\text{baseline}} = \max_k \hat p_k$$

where $\hat p_k$ is the training class frequency. Under imbalance, this can be high while F1 on the minority class is zero (Section 21).

**DummyRegressor** and **DummyClassifier** in scikit-learn implement these baselines for quick comparison.

### What "beating the baseline" means

| Task | Baseline | Model adds value when |
|------|----------|----------------------|
| Regression | Mean / median | $\text{MSE}_{\text{model}} < \text{MSE}_{\text{baseline}}$ or $R^2 > 0$ |
| Classification | Majority class | Accuracy **and** minority-class recall/F1 exceed baseline |
| Ranking | Random order | ROC-AUC $> 0.5$ or PR-AUC $> \pi$ |

A small improvement on accuracy alone may not matter if the business metric is minority recall.

### Interpreting comparisons

- If model and baseline scores are close, the difference may be **noise**. With small test sets, a few misclassified points move metrics sharply.
- Always report **class prevalence** alongside classification metrics.
- A model that beats the baseline on average but fails on important subgroups is not necessarily deployable.

Below: majority-class baseline vs balanced logistic regression on the imbalanced split from Section 21.


In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_tr, y_tr)
learned = LogisticRegression(max_iter=2000, class_weight="balanced")
learned.fit(X_tr, y_tr)

rows = []
for name, m in [("Majority baseline", baseline), ("Logistic (balanced)", learned)]:
    pred = m.predict(X_te)
    rows.append({
        "model": name,
        "accuracy": accuracy_score(y_te, pred),
        "precision": precision_score(y_te, pred, zero_division=0),
        "recall": recall_score(y_te, pred, zero_division=0),
        "F1": f1_score(y_te, pred, zero_division=0),
    })

for r in rows:
    print(
        f"{r['model']:22s}  acc={r['accuracy']:.3f}  "
        f"prec={r['precision']:.3f}  rec={r['recall']:.3f}  F1={r['F1']:.3f}"
    )


## 23. Introduction to Interpretability

### What interpretability provides

**Evaluation metrics** (Section 16) summarize how well a model predicts. **Interpretability** methods address a different question: **which inputs drive predictions, and in what direction?** This supports debugging, trust, and compliance — but interpretability output is **not** causal proof (Section 4).

Two scopes:

| Scope | Question | Examples |
|-------|----------|----------|
| **Global** | What features matter overall? | Coefficients, permutation importance, PDP |
| **Local** | Why this single prediction? | SHAP for one row, LIME (deferred) |

### Linear models: coefficient interpretation

For logistic regression with logit link:

$$\log \frac{\hat p}{1-\hat p} = \theta_0 + \sum_j \theta_j x_j$$

Holding other features fixed, increasing $x_j$ by one unit changes the **log-odds** of the positive class by $\theta_j$. Positive $\theta_j$ pushes toward class 1; negative toward class 0. **Magnitude** $|θ_j|$ indicates strength only when features are on comparable scales (hence scaling in Section 18).

**Caveat:** correlated features split credit arbitrarily between coefficients; individual $\theta_j$ should not be read as isolated causal effects.

### Tree-based importance (Gini / impurity decrease)

At each split, a feature reduces node impurity (Gini or entropy). **Feature importance** sums impurity decrease attributable to each feature across all trees. Features used early and often rank higher.

**Limitations:** biased toward high-cardinality or many-valued features; does not measure interaction effects cleanly; values are relative, not calibrated probabilities.

### Permutation importance

For feature $j$, permute column $x_j$ in the validation set (destroying its relationship with $y$) while keeping other columns fixed. The **importance** is the drop in score:

$$I_j = \text{metric}(f, X, y) - \text{metric}(f, X^{(\pi_j)}, y)$$

where $X^{(\pi_j)}$ has column $j$ randomly shuffled. Larger $I_j$ means the model relied on feature $j$.

**Properties:**

- Model-agnostic (works for any fitted predictor).
- Uses **held-out** data; no retraining required.
- Correlated features can share importance: permuting one may leave signal in a duplicate column.

**Not causal:** $I_j > 0$ means the model **uses** $x_j$, not that $x_j$ **causes** $y$.

### Partial dependence plots (PDP)

The **partial dependence** of feature $j$ averages predictions while varying $x_j$ and marginalizing over other features:

$$\text{PD}_j(x_j) = \mathbb E_{x_{-j}}\bigl[ f(x_j, x_{-j}) \bigr]$$

The plot shows the average model response along one dimension. **ICE plots** (individual conditional expectation) show the same for single rows.

### SHAP (brief)

**SHAP** values attribute each prediction to features using cooperative game theory (Shapley values). For a row $x$, $\phi_j$ is the contribution of feature $j$ to $\hat f(x) - \mathbb E[\hat f]$. Local attributions sum to the prediction deviation from the mean.

SHAP is more expensive but gives consistent local explanations; covered in later material.

### When to use which tool

| Tool | Best for | Avoid when |
|------|----------|------------|
| Linear coefficients | Linear/logistic models, signed direction | Features strongly correlated |
| Tree impurity importance | Quick forest screening | Need unbiased ranking |
| Permutation importance | Any model, validation-based ranking | Features duplicated / leaky |
| PDP | Nonlinear average effect of one feature | Strong feature interactions |
| SHAP | Local explanations with additivity | Large datasets without approximation |

Below: permutation importance on the logistic model from Section 21.


In [ ]:
from sklearn.inspection import permutation_importance

clf = LogisticRegression(max_iter=2000, class_weight="balanced").fit(X_tr, y_tr)
perm = permutation_importance(
    clf, X_te, y_te, n_repeats=10, random_state=0, scoring="f1",
)
top = np.argsort(perm.importances_mean)[-8:]

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(
    [f"feature {i}" for i in top],
    perm.importances_mean[top],
    xerr=perm.importances_std[top],
    color="C0",
)
ax.set_xlabel("F1 drop when feature is permuted")
ax.set_title("Permutation importance (higher = model relied more on feature)")
plt.tight_layout()
plt.show()

print("Top feature indices:", top[::-1])
